In [7]:
from datasets import load_dataset
from transformers import pipeline
from sklearn.metrics import accuracy_score, f1_score
import pandas as pd
import re

In [8]:
dataset = load_dataset("stanfordnlp/imdb", split="test")
sample = dataset.shuffle(seed=42).select(range(500))
texts = list(sample["text"])
labels = list(sample["label"])

Device set to use mps:0


In [17]:
for r in range(50):
    classifier = pipeline("sentiment-analysis", model="lvwerra/distilbert-imdb")
    tokenizer = classifier.tokenizer
    print(len(tokenizer.encode(texts[r])))

Device set to use mps:0


437


Device set to use mps:0


240


Device set to use mps:0


...

264


Device set to use mps:0


368


Device set to use mps:0


165


In [ ]:
def clean(text):
    return re.sub(r"<br\s*/?>", " ", text)

To test this, I tried keeping the first 128 and the last 382 tokens of long reviews, following the head + tail strategy from [Sun et al. (2019)](https://arxiv.org/abs/1905.05583), which performed best on IMDB in their experiments.

In [9]:
def head_tail(text, tokenizer, max_len=510, head=128):
    ids = tokenizer.encode(text, add_special_tokens=False)
    if len(ids) <= max_len:
        return text
    ids = ids[:head] + ids[-(max_len - head):]
    return tokenizer.decode(ids)

In [10]:
def evaluate(classifier, use_head_tail):
    tokenizer = classifier.tokenizer
    if use_head_tail:
        inputs = [head_tail(clean(t), tokenizer) for t in texts]
    else:
        inputs = [clean(t) for t in texts]

    results = classifier(inputs, truncation=True, batch_size=16)
    label_map = {"NEGATIVE": 0, "POSITIVE": 1}
    preds = [label_map[r["label"]] for r in results]
    return preds

In [11]:
models = {
    "SST-2": "distilbert/distilbert-base-uncased-finetuned-sst-2-english",
    "IMDB": "lvwerra/distilbert-imdb",
}

rows = []
all_preds = {}

for name, model_id in models.items():
    classifier = pipeline("sentiment-analysis", model=model_id)
    for use_ht in [False, True]:
        preds = evaluate(classifier, use_ht)
        setting = "head + tail" if use_ht else "truncate"
        all_preds[(name, setting)] = preds
        rows.append({
            "Model": name,
            "Preprocessing": setting,
            "Accuracy": accuracy_score(labels, preds),
            "F1": f1_score(labels, preds),
        })

results_df = pd.DataFrame(rows)
results_df

Device set to use mps:0
Token indices sequence length is longer than the specified maximum sequence length for this model (590 > 512). Running this sequence through the model will result in indexing errors
Device set to use mps:0
Token indices sequence length is longer than the specified maximum sequence length for this model (590 > 512). Running this sequence through the model will result in indexing errors


,Model,Preprocessing,Accuracy,F1
0,SST-2,truncate,0.892,0.888889
1,SST-2,head + tail,0.884,0.881148
2,IMDB,truncate,0.926,0.924949
3,IMDB,head + tail,0.920,0.919028


In [14]:
preds = all_preds[("IMDB", "truncate")]

errors = [(t, y, p) for t, y, p in zip(texts, labels, preds) if y != p]
for t, y, p in errors[:5]:
    print(f"label {y} / pred {p}: {t[:300]}...\n")

label 0 / pred 1: A holiday on a boat, a married couple, an angry waiter and a shipwreck is the reason to this films beginning.<br /><br />I like boobs. No question about that. But when the main character allies with whoever happens to have the most fish at the moment, mostly by having sex with them and playing the r...

label 0 / pred 1: Coming from Kiarostami, this art-house visual and sound exposition is a surprise. For a director known for his narratives and keen observation of humans, especially children, this excursion into minimalist cinematography begs for questions: Why did he do it? Was it to keep him busy during a vacation...

label 1 / pred 0: Sex, drugs, racism and of course you ABC's. What more could you want in a kid's show!<br /><br />------------------------------------------- -------------------------------------------<br /><br />"User Comment Guidelines <br /><br />Please note there is a 1,000 word limit on comments. The recommende...

label 0 / pred 1: Clearly this 